# Code for Chapter 14 @ SSTC-NEU modified by Fu
## Combining Encoder with Decoder Transformers

In [ ]:
# %%capture 
## NOTE: Installation might take a couple of minutes, so go get yourself a snack!
## %%capture prevents this cell from printing a ton of STDERR stuff to the screen

!pip install bertopic datasets datamapplot WordCloud
## bertopic - BERTopic is a BERT based LLM that combines transformers and c-TF-IDF 
##            to create clusters of easily interpretable topics
##            NOTE: By default, bertopic comes with sentence-transformers (aka SBERT), UMAP and HDBSCAN
##                  because those, plus C-TF-IDF, form the standard BERTopic pipeline.
## datasets - a huggingface package that makes it easy to access datasets stored on their site
## datamapplot - creates nice looking plots of data maps
## WordCloud - generates word clouds

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 224.3 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 470.2 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.6/342.6 kB 556.7 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.2/526.2 kB 652.4 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 1.1 MB/s eta 0:00:0000:0100:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 2.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 9.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 7.5 MB/s eta 0:00:0000:0100:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 7.1 MB/s eta 0:00:0000:0100:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 27.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━

In [ ]:
# Load the data directly from huggingface using their datasets module
from datasets import load_dataset

dataset = load_dataset("maartengr/arxiv_nlp")["train"]
dataset

ModuleNotFoundError: No module named 'datasets'

In [ ]:
abstracts = dataset["Abstracts"]
titles = dataset["Titles"]

In [ ]:
abstracts[0:2]

In [ ]:
from sentence_transformers import SentenceTransformer

# download the pre-trained SBERT model
embedding_model = SentenceTransformer('thenlper/gte-small')

## create the embeddings for the abstracts.
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

In [ ]:
# Check the dimensions of the resulting embeddings
embeddings.shape

In [ ]:
from umap import UMAP

# We reduce the input embeddings from 384 dimenions to 5 dimenions
umap_model = UMAP(
    n_components=5, 
    min_dist=0.0, 
    metric='cosine', 
    random_state=42
)
reduced_embeddings_5 = umap_model.fit_transform(embeddings)

In [ ]:
# Check the dimensions of the reduced embeddings
reduced_embeddings_5.shape

In [ ]:
from hdbscan import HDBSCAN

## NOTE: The HDBSCAN call will generate a bunch of warnings. 
## If those bother you, you can run this code:
# import os
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Find and label clusters
hdbscan_model = HDBSCAN(
    min_cluster_size=50, # each cluster must have >= 50 documents in it
    metric='euclidean',  # clustering is based on euclidean distance
    cluster_selection_method='eom' # eom = Excess of Mass and is the default clustering selection method
).fit(reduced_embeddings_5)
clusters = hdbscan_model.labels_

In [ ]:
# Print out the number of cluseters
len(set(clusters)) # set() removes duplicates - so we only count the unique cluster labels

In [ ]:
import numpy as np

# Print first three documents in cluster 0
cluster = 0
for index in np.where(clusters==cluster)[0][:3]:
    print(abstracts[int(index)][:300] + "... \n")

In [ ]:
import pandas as pd

# Reduce 384-dimensional embeddings to 2 dimensions for easier visualization
reduced_embeddings_2 = UMAP(
    n_components=2, 
    min_dist=0.0, 
    metric='cosine', 
    random_state=42
).fit_transform(embeddings)

# Create dataframe
df = pd.DataFrame(reduced_embeddings_2, columns=["x", "y"])
df["title"] = titles
df["cluster"] = [str(c) for c in clusters]

# Select outliers and non-outliers (clusters)
clusters_df = df.loc[df.cluster != "-1", :]
outliers_df = df.loc[df.cluster == "-1", :]

In [ ]:
import matplotlib.pyplot as plt

# Plot outliers and non-outliers seperately
plt.scatter(outliers_df.x, outliers_df.y, alpha=0.05, s=2, c="grey")
plt.scatter(
    clusters_df.x, clusters_df.y, c=clusters_df.cluster.astype(int),
    alpha=0.6, s=2, cmap='tab20b'
)
plt.axis('off')
# plt.savefig("matplotlib.png", dpi=300)  # Uncomment to save the graph as a .png

In [ ]:
from bertopic import BERTopic

# Train our model with our previously defined models
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
).fit(abstracts, embeddings)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(0)

In [ ]:
topic_model.find_topics("topic modeling")

In [ ]:
topic_model.topics_[titles.index('BERTopic: Neural topic modeling with a class-based TF-IDF procedure')]

In [ ]:
## Visualize topics and documents
## NOTE: THIS WORKS ON GOOGLE COLAB BUT NOT ON RUNPOD...
fig = topic_model.visualize_documents(
    list(titles),
    reduced_embeddings=reduced_embeddings_2,
    width=1200,
    hide_annotations=True
)

# Update fonts of legend for easier visualization
fig.update_layout(font=dict(size=16))

In [ ]:
# Save original representations
from copy import deepcopy
original_topics = deepcopy(topic_model.topic_representations_)

In [ ]:
def topics_vs_labels(original_topics, model, nr_topics=5):
    """Show the differences in topics and the new labels"""
    df = pd.DataFrame(columns=["Topic ID", "Original Topics", "New Labels"])
    for topic in range(nr_topics):

        # Extract top 5 words from the original topic
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = model.get_topic(topic)[0][0]
        df.loc[len(df)] = [topic, og_words, new_words]

    return df

In [ ]:
from transformers import pipeline
from bertopic.representation import TextGeneration

prompt = """I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the documents and keywords, what is this topic about?"""

# Update our topic representations using Flan-T5
generator = pipeline('text2text-generation', model='google/flan-t5-small', device=0)
representation_model = TextGeneration(
    generator, prompt=prompt, doc_length=50, tokenizer="whitespace"
)
topic_model.update_topics(abstracts, representation_model=representation_model)

In [ ]:
# compare the original topics with the new labels
topics_vs_labels(original_topics, topic_model)

In [ ]:
# Visualize documents with labeled topics
fig = topic_model.visualize_document_datamap(
    titles,
    topics=list(range(20)),
    reduced_embeddings=reduced_embeddings_2,
    width=1200
)
plt.savefig("datamapplot.png", dpi=300)

In [ ]:
topic_model.update_topics(abstracts, top_n_words=500)

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def create_wordcloud(model, topic):
    plt.figure(figsize=(10,5))
    text = {word: value for word, value in model.get_topic(topic)}
    wc = WordCloud(background_color="white", max_words=1000, width=1600, height=800)
    wc.generate_from_frequencies(text)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.show()

# Show wordcloud
create_wordcloud(topic_model, topic=17)